# Module Import Test Notebook

This notebook validates that the refactored modules load correctly.

In [1]:
#========================================
# Setup paths and notebook defaults
#========================================
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as pp
import seaborn as sns
from supabase import create_client
import os
from dotenv import load_dotenv

BASE_DIR = Path.cwd().resolve().parent
DATA_DIR = BASE_DIR / "data"
INPUT_DIR = DATA_DIR / "input"


if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"BASE_DIR: {BASE_DIR}")

BASE_DIR: C:\Users\F.Turner\Documents\MwM Proof of Concept V2


In [2]:
from src.ingest import *
from src.aggregate import *
from src.combine import *
from src.build_cmos import *
from src.theme import *

print("Imports succeeded: ingest.py, aggregate.py, combine.py, output.py, build_cmos.py, and theme.py are available.")

Imports succeeded: ingest.py, aggregate.py, combine.py, output.py, build_cmos.py, and theme.py are available.


In [3]:
def run_pipeline(BASE_URL=None, ASSET_ID=None, API_KEY=None, file_path=None,
                 metadata_overrides=None, interactive=True,
                 drivers=None, cmo_level="school", finalize=True):
    """Run ingest -> aggregate -> combine -> build_cmos, updating Supabase at each stage.

    Supply either file_path or (BASE_URL, ASSET_ID, API_KEY). Updates import_log,
    combined_<level>_level, indicator_values, cio_table and narratives.
    finalize=False loads the form only (no school rollup or CMOs); use it for all but the
    last form of a batch, then the final form (finalize=True) rolls up and builds CMOs once.
    Returns the aggregated dataset and a dict of {driver_id: (long_table, cio_table, narratives)}.
    """
    load_dotenv()
    client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

    print("1/4 Ingest...")
    dataset = build_dataset(BASE_URL=BASE_URL, ASSET_ID=ASSET_ID, API_KEY=API_KEY, file_path=file_path,
                            metadata_overrides=metadata_overrides, interactive=interactive)

    print("2/4 Aggregate...")
    dataset = aggregates(dataset)

    print("3/4 Combine...")
    combine(dataset, rollup=finalize)

    if not finalize:
        print("Loaded (finalize=False): rollup and CMOs skipped.")
        return dataset, {}

    print("4/4 Build CMOs...")
    country = dataset.metadata.get("country")
    if not country or country == "unknown":
        raise ValueError("dataset.metadata['country'] is required to build CMO tables.")
    results = {}
    if drivers is None:
        drivers = list_drivers(client)
        if not drivers:
            raise ValueError("No drivers found in the 'drivers' table; pass drivers=... explicitly.")
    for driver_id in drivers:
        try:
            results[driver_id] = build_cmos(country=str(country).lower(), level=cmo_level,
                                            driver_id=driver_id, client=client)
            print(f"  {driver_id}: done")
        except ValueError as e:
            print(f"  {driver_id}: skipped ({e})")

    print("Pipeline complete.")
    return dataset, results


def split_files_by_country(file_paths):
    """Split each CSV/Excel file into one file per country, saved alongside the original.

    Output files are named <stem>_<country><suffix>. Files with a single country (or no
    country column) are not split and their original path is returned. Returns a list of
    file paths to pass to run_pipeline.
    """
    new_paths = []
    for path in map(Path, file_paths):
        df = import_data(str(path))
        country_col = next((c for c in df.columns if str(c).split("/")[-1].strip().lower() == "country"), None)
        countries = df[country_col].dropna().astype(str).str.strip().unique() if country_col else []
        if len(countries) <= 1:
            new_paths.append(path)
            continue
        for country in countries:
            part = df[df[country_col].astype(str).str.strip() == country]
            safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in country.lower())
            out = path.with_name(f"{path.stem}_{safe}{path.suffix}")
            if path.suffix.lower() == ".csv":
                part.to_csv(out, index=False)
            else:
                part.to_excel(out, index=False)
            new_paths.append(out)
        print(f"{path.name}: split into {len(countries)} files ({', '.join(countries)})")
    return new_paths

In [4]:
INPUT = Path(INPUT_DIR).parent.parent / "synthetic_v2" / "data"
countries = ["syria", "sudan"]
# Process child and teacher data before school, since school_combined aggregates them
# (level, file stem, form number)
forms = [
    ("child", "child_assessments", "01"),
    ("child", "child_monthly", "02"),
    ("teacher", "teacher_monthly", "01"),
    ("school", "school_monthly", "01"),
]

jobs = []
for country in countries:
    for level, stem, number in forms:
        is_master = stem.endswith("_master")
        jobs.append((
            INPUT / f"{stem}_{country}.csv",
            {
                "country": country,
                "form_id": f"{country}_{level}_{number}",
                "level": level,
                "survey_name": f"{stem}_{country}",
                "first_submission": "2025-09-01",
                "last_submission": "2025-09-01" if is_master else "2026-08-31",
            },
        ))

failures = {}
import time
timings = {}
batch_start = time.perf_counter()
for i, (path, overrides) in enumerate(jobs):
    # only the last job per country rolls up to school level and builds CMOs
    last_for_country = all(j[1]["country"] != overrides["country"] for j in jobs[i + 1:])
    print(f"\n##### {overrides['form_id']}  ({path.name})")
    job_start = time.perf_counter()
    try:
        run_pipeline(file_path=str(path), metadata_overrides=overrides, interactive=False,
                     cmo_level="school",
                     finalize=last_for_country)
    except Exception as e:
        failures[overrides["form_id"]] = repr(e)
        print(f"FAILED: {e!r}")
    timings[overrides["form_id"]] = time.perf_counter() - job_start
    print(f"--> {overrides['form_id']} took {timings[overrides['form_id']]:.1f}s "
          f"(total elapsed {time.perf_counter() - batch_start:.1f}s)")

print("\nFailures:", failures or "none")
print(f"Total time: {time.perf_counter() - batch_start:.1f}s")


##### syria_child_01  (child_assessments_syria.csv)
1/4 Ingest...
Form ID syria_child_01 already exists in the log. Processed data will be added to the existing dataset.
Metadata added to log:
{'form_id': 'syria_child_01', 'survey_name': 'child_assessments_syria', 'level': 'child', 'first_submission': '2025-09-15 00:00:00', 'last_submission': '2026-03-15 00:00:00', 'rows': 1152, 'country': 'Syria', '_uuid_cols': 'country,child_id,round', 'logged_at': datetime.datetime(2026, 10, 13, 17, 39)}
Data imported from file. Kobo metadata functions will be skipped (question dataframe and choices dataframe). Only basic metadata will be logged.
UUID key columns: ['country', 'child_id', 'round']
2/4 Aggregate...
Available dimensions: ['class_id', 'teacher_id', 'child_id', 'sex', 'age', 'age_group', 'round', 'year_month', 'displacement_status', 'disability', 'date_time', 'admin0', 'admin1', 'school_id']
Missing dimensions: []
New columns added: ['age_group', 'disability', 'date_time', 'admin0']
Ava

In [1]:
import numpy as np
import plotly.graph_objects as go

# Example data
data = {
    "UL": {"schools": 40, "color": "#c9dfc1"},  # Upper Left
    "UR": {"schools": 20, "color": "#4ea72e"},  # Upper Right
    "LL": {"schools": 15, "color": "#c65911"},  # Lower Left
    "LR": {"schools": 35, "color": "#f1dfd3"},  # Lower Right
}

# Convert counts to side lengths
max_side = 4

max_schools = max(v["schools"] for v in data.values())

for q in data:
    data[q]["side"] = np.sqrt(data[q]["schools"] / max_schools) * max_side

fig = go.Figure()

# UL
s = data["UL"]["side"]
fig.add_shape(
    type="rect",
    x0=-s, y0=0,
    x1=0,  y1=s,
    fillcolor=data["UL"]["color"],
    line_width=0
)

# UR
s = data["UR"]["side"]
fig.add_shape(
    type="rect",
    x0=0, y0=0,
    x1=s, y1=s,
    fillcolor=data["UR"]["color"],
    line_width=0
)

# LL
s = data["LL"]["side"]
fig.add_shape(
    type="rect",
    x0=-s, y0=-s,
    x1=0,  y1=0,
    fillcolor=data["LL"]["color"],
    line_width=0
)

# LR
s = data["LR"]["side"]
fig.add_shape(
    type="rect",
    x0=0, y0=-s,
    x1=s, y1=0,
    fillcolor=data["LR"]["color"],
    line_width=0
)

# Central axes
fig.add_shape(
    type="line",
    x0=-5, x1=5,
    y0=0, y1=0,
    line=dict(color="#0b5d80", width=4)
)

fig.add_shape(
    type="line",
    x0=0, x1=0,
    y0=-5, y1=5,
    line=dict(color="#0b5d80", width=4)
)

# Labels
fig.add_annotation(
    x=-data["UL"]["side"]/2,
    y=data["UL"]["side"]/2,
    text="Low Quality<br>High Frequency<br><b>40</b>",
    showarrow=False
)

fig.add_annotation(
    x=data["UR"]["side"]/2,
    y=data["UR"]["side"]/2,
    text="High Quality<br>High Frequency<br><b>20</b>",
    showarrow=False
)

fig.add_annotation(
    x=-data["LL"]["side"]/2,
    y=-data["LL"]["side"]/2,
    text="Low Quality<br>Low Frequency<br><b>15</b>",
    showarrow=False
)

fig.add_annotation(
    x=data["LR"]["side"]/2,
    y=-data["LR"]["side"]/2,
    text="High Quality<br>Low Frequency<br><b>35</b>",
    showarrow=False
)

fig.update_layout(
    width=1000,
    height=700,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=20, b=20),
)

fig.update_xaxes(
    visible=False,
    range=[-5, 5],
    fixedrange=True
)

fig.update_yaxes(
    visible=False,
    range=[-5, 5],
    scaleanchor="x",
    fixedrange=True
)

fig.show()